<a href="https://colab.research.google.com/github/neohack22/ebw3nt/blob/main/apprentissage/correction_classification_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation des dépendances
Installez (ou mettez à jour) les bibliothèques nécessaires pour ce TP : `keras` et `keras-hub`.

In [ ]:
!pip install keras keras-hub --upgrade -q

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

In [ ]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

### Charger le dataset IMDb
Chargez le dataset IMDb depuis `keras.datasets.imdb` en ne conservant que les **10 000 mots** les plus fréquents, puis récupérez `(train_data, train_labels)` et `(test_data, test_labels)`.

In [ ]:
from keras.datasets import imdb

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(
    num_words=10000
)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


### Inspecter une critique et son label
Affichez la première critique de l’ensemble d’entraînement (`train_data[0]`) et son label (`train_labels[0]`).

In [ ]:
train_data[0]

[1,
 14,
 22,
 16,
 43,
 530,
 973,
 1622,
 1385,
 65,
 458,
 4468,
 66,
 3941,
 4,
 173,
 36,
 256,
 5,
 25,
 100,
 43,
 838,
 112,
 50,
 670,
 2,
 9,
 35,
 480,
 284,
 5,
 150,
 4,
 172,
 112,
 167,
 2,
 336,
 385,
 39,
 4,
 172,
 4536,
 1111,
 17,
 546,
 38,
 13,
 447,
 4,
 192,
 50,
 16,
 6,
 147,
 2025,
 19,
 14,
 22,
 4,
 1920,
 4613,
 469,
 4,
 22,
 71,
 87,
 12,
 16,
 43,
 530,
 38,
 76,
 15,
 13,
 1247,
 4,
 22,
 17,
 515,
 17,
 12,
 16,
 626,
 18,
 2,
 5,
 62,
 386,
 12,
 8,
 316,
 8,
 106,
 5,
 4,
 2223,
 5244,
 16,
 480,
 66,
 3785,
 33,
 4,
 130,
 12,
 16,
 38,
 619,
 5,
 25,
 124,
 51,
 36,
 135,
 48,
 25,
 1415,
 33,
 6,
 22,
 12,
 215,
 28,
 77,
 52,
 5,
 14,
 407,
 16,
 82,
 2,
 8,
 4,
 107,
 117,
 5952,
 15,
 256,
 4,
 2,
 7,
 3766,
 5,
 723,
 36,
 71,
 43,
 530,
 476,
 26,
 400,
 317,
 46,
 7,
 4,
 2,
 1029,
 13,
 104,
 88,
 4,
 381,
 15,
 297,
 98,
 32,
 2071,
 56,
 26,
 141,
 6,
 194,
 7486,
 18,
 4,
 226,
 22,
 21,
 134,
 476,
 26,
 480,
 5,
 144,
 30,
 5535,
 18,

In [ ]:
train_labels[0]

np.int64(1)

### Vérifier l’index maximal des mots
Calculez l’indice de mot maximal présent dans l’ensemble d’entraînement (pour vérifier que l’on reste bien dans la limite des 10 000 mots).

In [ ]:
max([max(sequence) for sequence in train_data])

### Décoder une critique en texte
Récupérez le dictionnaire `word_index` d’IMDb, inversez-le pour obtenir un mapping index→mot, puis reconstruisez le texte de `train_data[0]` sous forme d’une chaîne de caractères.

### Afficher un extrait du texte décodé
Affichez les ~100 premiers caractères du texte décodé.

In [ ]:
word_index = imdb.get_word_index()
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])
decoded_review = " ".join(
    [reverse_word_index.get(i - 3, "?") for i in train_data[0]]
)

In [ ]:
decoded_review[:100]

### Encoder les séquences en représentation multi-hot
Implémentez une fonction `multi_hot_encode(sequences, num_classes)` qui transforme une liste de séquences d’indices en une matrice binaire (multi-hot) de taille `(nb_exemples, num_classes)`.
Utilisez-la pour créer `x_train` et `x_test` avec `num_classes=10000`.

In [ ]:
import numpy as np

def multi_hot_encode(sequences, num_classes):
    results = np.zeros((len(sequences), num_classes))
    for i, sequence in enumerate(sequences):
        results[i][sequence] = 1.0
    return results

x_train = multi_hot_encode(train_data, num_classes=10000)
x_test = multi_hot_encode(test_data, num_classes=10000)

### Vérifier l’encodage
Affichez le vecteur multi-hot correspondant au premier exemple `x_train[0]`.

In [ ]:
x_train[0]

### Convertir les labels au bon type
Convertissez `train_labels` et `test_labels` en `float32` pour obtenir `y_train` et `y_test`.

### Convertir les labels au bon type
Convertissez `train_labels` et `test_labels` en `float32` pour obtenir `y_train` et `y_test`.

In [ ]:
y_train = train_labels.astype("float32")
y_test = test_labels.astype("float32")

### Construire un modèle MLP pour classification binaire
Créez un modèle `keras.Sequential` composé de :
- 2 couches `Dense(16, relu)`
- 1 couche de sortie `Dense(1, sigmoid)`

In [ ]:
import keras
from keras import layers

model = keras.Sequential(
    [
        layers.Dense(16, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ]
)

### Compiler le modèle
Compilez le modèle avec :
- optimiseur `adam`
- loss `binary_crossentropy`
- métrique `accuracy`

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

### Créer un ensemble de validation
Créez un jeu de validation avec les **10 000 premiers exemples** de `x_train`/`y_train`, et gardez le reste comme jeu d’entraînement partiel.

In [ ]:
x_val = x_train[:10000]
partial_x_train = x_train[10000:]
y_val = y_train[:10000]
partial_y_train = y_train[10000:]

### Entraîner le modèle avec validation explicite
Entraînez le modèle sur le jeu d’entraînement partiel pendant 20 epochs, batch_size=512, en passant `(x_val, y_val)` comme données de validation. Stockez l’historique dans `history`.

In [ ]:
history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
)

### Variante : entraînement avec `validation_split`
Entraînez (ou réentraînez) le modèle sur `x_train, y_train` avec 20 epochs, batch_size=512, et `validation_split=0.2`. Stockez l’historique.

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=20,
    batch_size=512,
    validation_split=0.2,
)

### Explorer les métriques disponibles
Affichez les clés du dictionnaire `history.history` pour voir quelles courbes sont enregistrées.

In [ ]:
history_dict = history.history
history_dict.keys()

### Tracer la loss d’entraînement et de validation
À partir de `history.history`, récupérez `loss` et `val_loss`, créez un axe `epochs`, puis tracez les deux courbes (avec légende, titres et labels d’axes).

In [ ]:
import matplotlib.pyplot as plt

history_dict = history.history
loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]
epochs = range(1, len(loss_values) + 1)
plt.plot(epochs, loss_values, "r--", label="Training loss")
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.title("[IMDB] Training and validation loss")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Loss")
plt.legend()
plt.show()

### Tracer l’accuracy d’entraînement et de validation
Même principe : récupérez `accuracy` et `val_accuracy` depuis `history.history`, puis tracez les deux courbes.

In [ ]:
plt.clf()
acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]
plt.plot(epochs, acc, "r--", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("[IMDB] Training and validation accuracy")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Accuracy")
plt.legend()
plt.show()

### Réentraîner un modèle puis l’évaluer sur le test
Reconstruisez le même modèle (16-16-sigmoid), compilez-le, entraînez-le 4 epochs (batch_size=512) sur tout `x_train, y_train`, puis évaluez-le sur `x_test, y_test`. Stockez le résultat dans `results`.

In [ ]:
model = keras.Sequential(
    [
        layers.Dense(16, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ]
)
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.fit(x_train, y_train, epochs=4, batch_size=512)
results = model.evaluate(x_test, y_test)

### Afficher les résultats d’évaluation
Affichez la variable `results`.

In [ ]:
results

### Prédire sur de nouvelles données
Utilisez `model.predict(x_test)` pour générer des prédictions (probabilités) sur les données de test.

In [ ]:
model.predict(x_test)

### Charger le dataset Reuters
Chargez le dataset Reuters depuis `keras.datasets.reuters` en ne gardant que les 10 000 mots les plus fréquents, puis récupérez train/test (données + labels).

In [ ]:
from keras.datasets import reuters

(train_data, train_labels), (test_data, test_labels) = reuters.load_data(
    num_words=10000
)

### Vérifier la taille des ensembles
Affichez le nombre d’exemples dans `train_data` puis dans `test_data`.

In [ ]:
len(train_data)

In [ ]:
len(test_data)

### Inspecter un exemple
Affichez la séquence `train_data[10]` et le label associé `train_labels[10]`.

In [ ]:
train_data[10]

### Décoder un article Reuters en texte
Récupérez `word_index`, inversez-le et reconstruisez le texte correspondant à `train_data[10]` sous forme de chaîne.

In [ ]:
word_index = reuters.get_word_index()
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])
decoded_newswire = " ".join(
    [reverse_word_index.get(i - 3, "?") for i in train_data[10]]
)

In [ ]:
train_labels[10]

### Vectoriser les entrées en multi-hot
Réutilisez `multi_hot_encode` pour transformer `train_data` et `test_data` en matrices `x_train` et `x_test` de dimension 10 000.

In [ ]:
x_train = multi_hot_encode(train_data, num_classes=10000)
x_test = multi_hot_encode(test_data, num_classes=10000)

### Encoder les labels en one-hot (version “maison”)
Écrivez une fonction `one_hot_encode(labels, num_classes=46)` qui transforme un vecteur de labels entiers en matrice one-hot, puis créez `y_train` et `y_test`.

In [ ]:
def one_hot_encode(labels, num_classes=46):
    results = np.zeros((len(labels), num_classes))
    for i, label in enumerate(labels):
        results[i, label] = 1.0
    return results

y_train = one_hot_encode(train_labels)
y_test = one_hot_encode(test_labels)

### Encoder les labels en one-hot (version Keras)
Refaites l’encodage des labels en utilisant `keras.utils.to_categorical` pour obtenir `y_train` et `y_test`.

In [ ]:
from keras.utils import to_categorical

y_train = to_categorical(train_labels)
y_test = to_categorical(test_labels)

### Construire un modèle pour classification multi-classes
Créez un modèle `Sequential` composé de :
- `Dense(64, relu)`
- `Dense(64, relu)`
- `Dense(46, softmax)`

In [ ]:
model = keras.Sequential(
    [
        layers.Dense(64, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(46, activation="softmax"),
    ]
)

### Compiler avec une métrique Top-3 accuracy
Créez une métrique `TopKCategoricalAccuracy(k=3, name="top_3_accuracy")`, puis compilez le modèle avec :
- `adam`
- `categorical_crossentropy`
- métriques : `accuracy` et `top_3_accuracy`

In [ ]:
top_3_accuracy = keras.metrics.TopKCategoricalAccuracy(
    k=3, name="top_3_accuracy"
)
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy", top_3_accuracy],
)

### Créer un ensemble de validation (Reuters)
Utilisez les 1000 premiers exemples de `x_train`/`y_train` comme validation, et le reste comme entraînement partiel.

In [ ]:
x_val = x_train[:1000]
partial_x_train = x_train[1000:]
y_val = y_train[:1000]
partial_y_train = y_train[1000:]

### Entraîner avec validation explicite
Entraînez le modèle pendant 20 epochs (batch_size=512) avec `validation_data=(x_val, y_val)` et conservez l’historique.

In [ ]:
history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
)

### Tracer les courbes de loss
Tracez `loss` et `val_loss` en fonction des epochs (avec titre, légende, etc.).

In [ ]:
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, "r--", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Loss")
plt.legend()
plt.show()

### Tracer les courbes d’accuracy
Tracez `accuracy` et `val_accuracy` en fonction des epochs.

In [ ]:
plt.clf()
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
plt.plot(epochs, acc, "r--", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Accuracy")
plt.legend()
plt.show()

### Tracer la top-3 accuracy
Tracez `top_3_accuracy` et `val_top_3_accuracy` en fonction des epochs.

In [ ]:
plt.clf()
acc = history.history["top_3_accuracy"]
val_acc = history.history["val_top_3_accuracy"]
plt.plot(epochs, acc, "r--", label="Training top-3 accuracy")
plt.plot(epochs, val_acc, "b", label="Validation top-3 accuracy")
plt.title("Training and validation top-3 accuracy")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Top-3 accuracy")
plt.legend()
plt.show()

### Réentraîner puis évaluer sur le test (Reuters)
Reconstruisez le même modèle, compilez-le, entraînez-le 9 epochs sur tout `x_train, y_train` (batch_size=512), puis évaluez sur `x_test, y_test`.

In [ ]:
model = keras.Sequential(
    [
        layers.Dense(64, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(46, activation="softmax"),
    ]
)
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(
    x_train,
    y_train,
    epochs=9,
    batch_size=512,
)
results = model.evaluate(x_test, y_test)

### Afficher les résultats Reuters
Affichez `results`.

In [ ]:
results

### Estimer une baseline “hasard”
Copiez `test_labels`, mélangez la copie aléatoirement, comparez-la à `test_labels` et calculez la proportion de совпncidences (accuracy due au hasard).

In [ ]:
import copy
test_labels_copy = copy.copy(test_labels)
np.random.shuffle(test_labels_copy)
hits_array = np.array(test_labels == test_labels_copy)
hits_array.mean()

### Générer des prédictions
Calculez `predictions = model.predict(x_test)`.

In [ ]:
predictions = model.predict(x_test)

### Vérifier la forme d’une prédiction
Affichez la dimension de `predictions[0]` (elle doit correspondre au nombre de classes).

In [ ]:
predictions[0].shape

### Vérifier que les probabilités somment à 1
Calculez la somme des composantes de `predictions[0]`.

In [ ]:
np.sum(predictions[0])

### Trouver la classe prédite
Affichez l’indice de la probabilité maximale dans `predictions[0]`.

In [ ]:
np.argmax(predictions[0])

### Utiliser des labels entiers et une loss sparse
Remplacez `y_train` et `y_test` par les labels entiers (`train_labels`, `test_labels`), puis recompilez le modèle avec la loss `sparse_categorical_crossentropy`.

In [ ]:
y_train = train_labels
y_test = test_labels

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

### Tester un goulot d’étranglement (couche trop petite)
Entraînez un modèle similaire, mais avec une couche cachée très petite (`Dense(4, relu)`) entre `Dense(64, relu)` et la sortie `Dense(46, softmax)`. Observez l’impact sur la validation.

In [ ]:
model = keras.Sequential(
    [
        layers.Dense(64, activation="relu"),
        layers.Dense(4, activation="relu"),
        layers.Dense(46, activation="softmax"),
    ]
)
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=128,
    validation_data=(x_val, y_val),
)

### Charger le dataset California Housing
Chargez le dataset `california_housing` (version "small") et récupérez `train_data`, `train_targets`, `test_data`, `test_targets`.

In [ ]:
from keras.datasets import california_housing

(train_data, train_targets), (test_data, test_targets) = (
    california_housing.load_data(version="small")
)

### Vérifier les dimensions
Affichez `train_data.shape` et `test_data.shape`, puis affichez `train_targets`.

In [ ]:
train_data.shape

In [ ]:
test_data.shape

In [ ]:
train_targets

### Normaliser les variables d’entrée
Calculez la moyenne et l’écart-type de `train_data` (par feature), puis standardisez `train_data` et `test_data` pour obtenir `x_train` et `x_test`.

In [ ]:
mean = train_data.mean(axis=0)
std = train_data.std(axis=0)
x_train = (train_data - mean) / std
x_test = (test_data - mean) / std

### Mettre à l’échelle les cibles
Divisez `train_targets` et `test_targets` par 100000 pour obtenir `y_train` et `y_test`.

In [ ]:
y_train = train_targets / 100000
y_test = test_targets / 100000

### Écrire une fonction `get_model()` pour la régression
Créez une fonction qui construit et compile un modèle `Sequential` :
- `Dense(64, relu)`
- `Dense(64, relu)`
- sortie `Dense(1)`
Compilation :
- optimizer `adam`
- loss `mean_squared_error`
- métrique `mean_absolute_error`

In [ ]:
def get_model():
    model = keras.Sequential(
        [
            layers.Dense(64, activation="relu"),
            layers.Dense(64, activation="relu"),
            layers.Dense(1),
        ]
    )
    model.compile(
        optimizer="adam",
        loss="mean_squared_error",
        metrics=["mean_absolute_error"],
    )
    return model

### Mettre en place une validation croisée K-fold (k=4)
Implémentez une boucle sur 4 folds :
- à chaque itération, créez `fold_x_val`, `fold_y_val`, `fold_x_train`, `fold_y_train`
- entraînez un nouveau modèle (via `get_model()`) pendant 50 epochs (batch_size=16, verbose=0)
- évaluez sur le fold de validation
- stockez la MAE de validation dans une liste `all_scores`

In [ ]:
k = 4
num_val_samples = len(x_train) // k
num_epochs = 50
all_scores = []
for i in range(k):
    print(f"Processing fold #{i + 1}")
    fold_x_val = x_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_y_val = y_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_x_train = np.concatenate(
        [x_train[: i * num_val_samples], x_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    fold_y_train = np.concatenate(
        [y_train[: i * num_val_samples], y_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    model = get_model()
    model.fit(
        fold_x_train,
        fold_y_train,
        epochs=num_epochs,
        batch_size=16,
        verbose=0,
    )
    scores = model.evaluate(fold_x_val, fold_y_val, verbose=0)
    val_loss, val_mae = scores
    all_scores.append(val_mae)

### Afficher les scores des folds
Affichez les MAE obtenues sur chaque fold (arrondies à 3 décimales), puis la moyenne des MAE.

In [ ]:
[round(value, 3) for value in all_scores]

In [ ]:
round(np.mean(all_scores), 3)

### Enregistrer l’historique des MAE en K-fold (200 epochs)
Refaites une validation croisée (k=4) mais en entraînant 200 epochs, en conservant à chaque fold la courbe `val_mean_absolute_error` dans `all_mae_histories`.

In [ ]:
k = 4
num_val_samples = len(x_train) // k
num_epochs = 200
all_mae_histories = []
for i in range(k):
    print(f"Processing fold #{i + 1}")
    fold_x_val = x_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_y_val = y_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_x_train = np.concatenate(
        [x_train[: i * num_val_samples], x_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    fold_y_train = np.concatenate(
        [y_train[: i * num_val_samples], y_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    model = get_model()
    history = model.fit(
        fold_x_train,
        fold_y_train,
        validation_data=(fold_x_val, fold_y_val),
        epochs=num_epochs,
        batch_size=16,
        verbose=0,
    )
    mae_history = history.history["val_mean_absolute_error"]
    all_mae_histories.append(mae_history)

### Tracer la MAE moyenne en fonction des epochs
Tracez `average_mae_history` (MAE moyenne) en fonction des epochs.

In [ ]:
average_mae_history = [
    np.mean([x[i] for x in all_mae_histories]) for i in range(num_epochs)
]

In [ ]:
epochs = range(1, len(average_mae_history) + 1)
plt.plot(epochs, average_mae_history)
plt.xlabel("Epochs")
plt.ylabel("Validation MAE")
plt.show()

### Tracer une version tronquée (à partir de l’epoch 10)
Ignorez les 10 premières epochs (phase instable) et tracez la MAE moyenne à partir de l’epoch 10.

In [ ]:
truncated_mae_history = average_mae_history[10:]
epochs = range(10, len(truncated_mae_history) + 10)
plt.plot(epochs, truncated_mae_history)
plt.xlabel("Epochs")
plt.ylabel("Validation MAE")
plt.show()

### Entraîner un modèle final et évaluer sur le test
Entraînez un nouveau modèle (`get_model()`) sur tout l’entraînement pendant 130 epochs (batch_size=16, verbose=0), puis évaluez-le sur le test et récupérez `test_mean_squared_error` et `test_mean_absolute_error`.

In [ ]:
model = get_model()
model.fit(x_train, y_train, epochs=130, batch_size=16, verbose=0)
test_mean_squared_error, test_mean_absolute_error = model.evaluate(
    x_test, y_test
)

In [ ]:
round(test_mean_absolute_error, 3)

### Générer des prédictions sur le test
Calculez `predictions = model.predict(x_test)` puis affichez la première prédiction `predictions[0]`.

In [ ]:
predictions = model.predict(x_test)
predictions[0]